In [1]:
import pandas as pd
import numpy as np
import json
import hashlib
from tqdm import tqdm

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import OneHotEncoder

tqdm.pandas()

# ==========================================================
# 1) JSONL LOADER
# ==========================================================
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


# ==========================================================
# 2) DESCRIPTION CLEANING + HASH USER ID
# ==========================================================
def extract_description(user_field):
    if isinstance(user_field, dict):
        d = user_field.get("description", "")
        return d if isinstance(d, str) else ""
    return ""

def clean_desc(s: str):
    s = s.strip().lower()
    return " ".join(s.split())

def to_day_index(dt):
    if pd.isna(dt):
        return -1
    ref = pd.Timestamp("2006-03-01")
    return (dt - ref).days

def build_pseudo_user_id(df):
    df = df.copy()

    # Description clean
    df["description_clean"] = df["user"].apply(extract_description).apply(clean_desc)

    # Parse account creation date
    df["user_created"] = pd.to_datetime(
        df["user"].apply(lambda u: u.get("created_at") if isinstance(u, dict) else None),
        errors="coerce"
    ).dt.tz_localize(None)

    df["creation_day"] = df["user_created"].apply(to_day_index)

    # Hash description + creation_day
    def hash_user(desc, day):
        return hashlib.md5(f"{desc}__{day}".encode()).hexdigest()

    df["pseudo_user_id"] = df.apply(
        lambda x: hash_user(x["description_clean"], x["creation_day"]),
        axis=1
    )

    return df


# ==========================================================
# 3) TWEET-LEVEL FEATURE EXTRACTION
# ==========================================================
def extract_all_features_tweet_level(df):
    """
    Extraction des features tweet-level :
    - user.* explosé en colonnes
    - dates => month_index + user_month_index + tweet_age_months
    - couleurs => booléens (is_default)
    - catégorielles => one-hot (lang + translator_type)
    - numériques / bool nettoyés
    """

    df = df.copy()

    # Explode user.*
    user_df = pd.json_normalize(df["user"])
    user_df.columns = [f"user.{c}" for c in user_df.columns]
    df = pd.concat([df, user_df], axis=1)

    NUMERIC_COLS = [
        "retweet_count", "favorite_count", "quote_count", "reply_count",
        "possibly_sensitive",
        "user.listed_count", "user.favourites_count", "user.statuses_count",
    ]

    BOOL_COLS = [
        "retweeted", "is_quote_status", "favorited", "truncated",
        "user.default_profile_image", "user.protected", "user.geo_enabled",
        "user.is_translator", "user.contributors_enabled",
        "user.profile_use_background_image", "user.profile_background_tile",
    ]

    CAT_COLS = ["user.translator_type", "user.lang"]

    # Dates
    DATE_TWEET = "created_at"
    DATE_USER  = "user.created_at"

    df[DATE_TWEET] = pd.to_datetime(df[DATE_TWEET], errors="coerce").dt.tz_localize(None)
    df[DATE_USER]  = pd.to_datetime(df[DATE_USER], errors="coerce").dt.tz_localize(None)

    REF_YEAR = 2006
    REF_MONTH = 3

    def to_month_index(dt):
        if pd.isna(dt):
            return -1
        return (dt.year - REF_YEAR) * 12 + (dt.month - REF_MONTH)

    df["tweet_month_index"] = df[DATE_TWEET].apply(to_month_index)
    df["user_month_index"]  = df[DATE_USER].apply(to_month_index)
    df["tweet_age_months"]  = df["tweet_month_index"] - df["user_month_index"]

    DATE_FEATURES = ["tweet_month_index", "user_month_index", "tweet_age_months"]

    # Default colors -> bool
    default_colors = {
        "user.profile_link_color": "1DA1F2",
        "user.profile_background_color": "F5F8FA",
        "user.profile_sidebar_border_color": "C0DEED",
        "user.profile_text_color": "333333",
        "user.profile_sidebar_fill_color": "DDEEF6",
    }

    for col, default_val in default_colors.items():
        df[col + "_is_default"] = (df[col] == default_val).astype(float)
        BOOL_COLS.append(col + "_is_default")

    df = df.drop(columns=list(default_colors.keys()), errors="ignore")

    # Clean numeric / bool
    for col in NUMERIC_COLS + DATE_FEATURES:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    for col in BOOL_COLS:
        df[col] = df[col].astype(float).fillna(0)

    # Categorical
    for col in CAT_COLS:
        df[col] = df[col].astype(str).fillna("missing")

    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    X_cat_arr = encoder.fit_transform(df[CAT_COLS])
    X_cat = pd.DataFrame(
        X_cat_arr,
        columns=encoder.get_feature_names_out(CAT_COLS),
        index=df.index
    )

    X = pd.concat([
        df[NUMERIC_COLS + DATE_FEATURES + BOOL_COLS],
        X_cat
    ], axis=1)

    return X


# ==========================================================
# 4) USER-LEVEL AGGREGATION
# ==========================================================
def aggregate_user_level(df, X_meta):
    """
    df      = df_train ou df_test (incluant pseudo_user_id + BERT scores)
    X_meta  = tweet-level features
    """

    df = df.copy()
    X = X_meta.copy()

    X["pseudo_user_id"] = df["pseudo_user_id"].values

    # BERT scores if present
    for col in ["bert_score", "bert_conf", "bert_descr_score", "bert_descr_conf"]:
        if col in df.columns:
            X[col] = df[col].values

    # BERT preds if présents (déjà renommés en bert_pred_text / bert_pred_desc)
    if "bert_pred_text" in df.columns:
        X["bert_pred_text"] = df["bert_pred_text"].values
    if "bert_pred_desc" in df.columns:
        X["bert_pred_desc"] = df["bert_pred_desc"].values

    # Numeric columns
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    num_cols = [c for c in num_cols if c != "pseudo_user_id"]

    agg_funcs = ["mean", "std", "min", "max"]
    grouped = X.groupby("pseudo_user_id")[num_cols].agg(agg_funcs)
    grouped.columns = [f"{col}_{stat}" for col, stat in grouped.columns]

    users = grouped.copy()

    # n_tweets
    tweet_counts = X.groupby("pseudo_user_id").size()
    users["n_tweets"] = tweet_counts
    users["log_n_tweets"] = np.log1p(tweet_counts)

    # BERT text stats
    if "bert_pred_text" in X.columns:
        grp = X.groupby("pseudo_user_id")["bert_pred_text"]
        users["bert_pred_text_1_count"] = grp.sum()
        users["bert_pred_text_0_count"] = users["n_tweets"] - users["bert_pred_text_1_count"]
        users["bert_pred_text_ratio"] = users["bert_pred_text_1_count"] / users["n_tweets"]

    # BERT desc stats
    if "bert_pred_desc" in X.columns:
        grp = X.groupby("pseudo_user_id")["bert_pred_desc"]
        users["bert_pred_desc_1_count"] = grp.sum()
        users["bert_pred_desc_0_count"] = users["n_tweets"] - users["bert_pred_desc_1_count"]
        users["bert_pred_desc_ratio"] = users["bert_pred_desc_1_count"] / users["n_tweets"]

    # Conditional means for bert_score
    if "bert_score" in X.columns:
        bert_cond = {}
        for uid, frame in X.groupby("pseudo_user_id"):
            mask1 = frame["bert_score"] > 0.5
            mask0 = ~mask1
            bert_cond[uid] = {
                "bert_score_mean_if1": frame["bert_score"][mask1].mean() if mask1.any() else 0,
                "bert_score_mean_if0": frame["bert_score"][mask0].mean() if mask0.any() else 0,
            }
        users = users.join(pd.DataFrame.from_dict(bert_cond, orient="index"))

    users = users.fillna(0)

    # Labels (si train)
    if "label" in df.columns:
        labels = df.groupby("pseudo_user_id")["label"].first()
        return users, labels
    else:
        return users


def build_user_level_dataset(df, mode="train"):
    """
    df : raw df with tweets (+ BERT merged)
    mode: "train" or "test"
    """
    df = build_pseudo_user_id(df)
    X_meta = extract_all_features_tweet_level(df)

    if mode == "train":
        X_user, y_user = aggregate_user_level(df, X_meta)
        return X_user, y_user
    else:
        X_user = aggregate_user_level(df, X_meta)
        return X_user


# ==========================================================
# 5) MAIN PIPELINE: MERGE BERT + BUILD USER DATASETS + XGB
# ==========================================================

# -------------------- LOAD TRAIN --------------------
print("Loading TRAIN jsonl...")
df_train = load_jsonl("/kaggle/input/influencers-or-observers-predicting-social-roles/Kaggle2025/train.jsonl")
df_train["challenge_id"] = df_train["challenge_id"].astype(str)

# -------------------- MERGE BERT OOF (TEXT + DESC) --------------------
print("Merging BERT OOF (TEXT + DESCRIPTION) on TRAIN...")

bert_oof_text = pd.read_csv("/kaggle/input/bert-with-proper-oof/bert_train_oof.csv")
bert_oof_desc = pd.read_csv("/kaggle/input/bert-with-proper-oof/bert_userdesc_train_oof.csv")

bert_oof_text["challenge_id"] = bert_oof_text["challenge_id"].astype(str)
bert_oof_desc["challenge_id"] = bert_oof_desc["challenge_id"].astype(str)

# Text BERT OOF (colonnes: challenge_id, bert_pred, prob_0, prob_1, bert_conf, label)
bert_oof_text = bert_oof_text.rename(columns={
    "prob_1": "bert_score",
    "bert_conf": "bert_conf",
    "bert_pred": "bert_pred_text"
})[["challenge_id", "bert_score", "bert_conf", "bert_pred_text"]]

# Desc BERT OOF (colonnes: challenge_id, pred, prob_0, prob_1, confidence, label)
bert_oof_desc = bert_oof_desc.rename(columns={
    "prob_1": "bert_descr_score",
    "confidence": "bert_descr_conf",
    "pred": "bert_pred_desc"
})[["challenge_id", "bert_descr_score", "bert_descr_conf", "bert_pred_desc"]]

df_train = df_train.merge(bert_oof_text, on="challenge_id", how="left")
df_train = df_train.merge(bert_oof_desc, on="challenge_id", how="left")

print("Missing TEXT OOF:", df_train["bert_score"].isna().sum())
print("Missing DESC OOF:", df_train["bert_descr_score"].isna().sum())

# Garder uniquement les lignes où on a des scores OOF (stacking propre)
df_train_oof = df_train.dropna(subset=["bert_score", "bert_descr_score"]).reset_index(drop=True)
print("Train OOF subset shape:", df_train_oof.shape)


# -------------------- BUILD USER-LEVEL TRAIN DATASET --------------------
print("\nBuilding USER-LEVEL TRAIN dataset...")
X_user_train, y_user_train = build_user_level_dataset(df_train_oof, mode="train")

print("X_user_train shape:", X_user_train.shape)
print("y_user_train shape:", y_user_train.shape)


# -------------------- XGBOOST TRAIN/VAL SPLIT --------------------
print("\nSplitting user-level TRAIN into train/val...")

X_tr, X_val, y_tr, y_val = train_test_split(
    X_user_train, y_user_train, test_size=0.2, random_state=42, stratify=y_user_train
)

dtrain = xgb.DMatrix(X_tr, label=y_tr)
dval   = xgb.DMatrix(X_val, label=y_val)

params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "eta": 0.03,
    "max_depth": 6,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
}

print("\nTraining XGBoost on USER-LEVEL data...")
model = xgb.train(
    params,
    dtrain,
    num_boost_round=2000,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=50,
    verbose_eval=50
)

print("\nBest iteration:", model.best_iteration)

y_val_pred_prob = model.predict(dval)
y_val_pred = (y_val_pred_prob > 0.5).astype(int)

print("\nUSER-LEVEL VAL Accuracy:", accuracy_score(y_val, y_val_pred))
print("\nClassification report:\n", classification_report(y_val, y_val_pred))


# ==========================================================
# 6) TEST PIPELINE: MERGE BERT + USER-LEVEL + PREDICTIONS
# ==========================================================

print("\nLoading TEST jsonl...")
df_test = load_jsonl("/kaggle/input/influencers-or-observers-predicting-social-roles/Kaggle2025/kaggle_test.jsonl")
df_test["challenge_id"] = df_test["challenge_id"].astype(str)

print("Merging BERT TEST predictions...")

bert_test_text = pd.read_csv("/kaggle/input/bert-with-proper-oof/bert_test_predictions.csv")
bert_test_desc = pd.read_csv("/kaggle/input/bert-with-proper-oof/bert_userdesc_test_predictions.csv")

bert_test_text["challenge_id"] = bert_test_text["challenge_id"].astype(str)
bert_test_desc["challenge_id"] = bert_test_desc["challenge_id"].astype(str)

# Text test: challenge_id, bert_pred, prob_0, prob_1, bert_conf
bert_test_text = bert_test_text.rename(columns={
    "prob_1": "bert_score",
    "bert_conf": "bert_conf",
    "bert_pred": "bert_pred_text"
})[["challenge_id", "bert_score", "bert_conf", "bert_pred_text"]]

# Desc test: challenge_id, pred, prob_0, prob_1, confidence
bert_test_desc = bert_test_desc.rename(columns={
    "prob_1": "bert_descr_score",
    "confidence": "bert_descr_conf",
    "pred": "bert_pred_desc"
})[["challenge_id", "bert_descr_score", "bert_descr_conf", "bert_pred_desc"]]

df_test = df_test.merge(bert_test_text, on="challenge_id", how="left")
df_test = df_test.merge(bert_test_desc, on="challenge_id", how="left")

# fill any remaining NaNs in BERT test
for col in ["bert_score", "bert_conf", "bert_descr_score", "bert_descr_conf"]:
    if col in df_test.columns:
        df_test[col] = df_test[col].fillna(0.5)

print("Test shape after BERT merge:", df_test.shape)


# -------------------- BUILD USER-LEVEL TEST DATASET --------------------
print("\nBuilding USER-LEVEL TEST dataset...")
X_user_test = build_user_level_dataset(df_test, mode="test")

print("X_user_test shape:", X_user_test.shape)

# Predict at USER level
dtest_users = xgb.DMatrix(X_user_test)
user_pred_prob = model.predict(dtest_users)
user_pred = (user_pred_prob > 0.5).astype(int)

df_user_pred = pd.DataFrame({
    "pseudo_user_id": X_user_test.index,
    "user_label": user_pred
})

# -------------------- MAP USER PREDICTIONS BACK TO TWEETS --------------------
df_test_users = build_pseudo_user_id(df_test)  # rebuild pseudo_user_id same way
df_test_users = df_test_users.merge(df_user_pred, on="pseudo_user_id", how="left")

submission = pd.DataFrame({
    "ID": df_test_users["challenge_id"],
    "label": df_test_users["user_label"].astype(int)
})

save_path = "/kaggle/working/submission_user_level_xgb.csv"
submission.to_csv(save_path, index=False)

print("\n🎉 Submission saved to:", save_path)
print(submission.head())


Loading TRAIN jsonl...
Merging BERT OOF (TEXT + DESCRIPTION) on TRAIN...
Missing TEXT OOF: 116120
Missing DESC OOF: 116120
Train OOF subset shape: (38794, 43)

Building USER-LEVEL TRAIN dataset...


/tmp/ipykernel_13/3462763141.py:52: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["user_created"] = pd.to_datetime(
/tmp/ipykernel_13/3462763141.py:110: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[DATE_TWEET] = pd.to_datetime(df[DATE_TWEET], errors="coerce").dt.tz_localize(None)
/tmp/ipykernel_13/3462763141.py:111: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[DATE_USER]  = pd.to_datetime(df[DATE_USER], errors="coerce").dt.tz_localize(None)
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `spar

X_user_train shape: (12722, 158)
y_user_train shape: (12722,)

Splitting user-level TRAIN into train/val...

Training XGBoost on USER-LEVEL data...
[0]	train-logloss:0.67398	val-logloss:0.67533
[50]	train-logloss:0.35990	val-logloss:0.40651
[100]	train-logloss:0.29394	val-logloss:0.36930
[150]	train-logloss:0.26685	val-logloss:0.36251
[200]	train-logloss:0.24994	val-logloss:0.36105
[250]	train-logloss:0.23553	val-logloss:0.36065
[298]	train-logloss:0.22132	val-logloss:0.36108

Best iteration: 248

USER-LEVEL VAL Accuracy: 0.8420432220039292

Classification report:
               precision    recall  f1-score   support

           0       0.84      0.87      0.86      1369
           1       0.84      0.81      0.83      1176

    accuracy                           0.84      2545
   macro avg       0.84      0.84      0.84      2545
weighted avg       0.84      0.84      0.84      2545


Loading TEST jsonl...
Merging BERT TEST predictions...
Test shape after BERT merge: (103380, 41)

Bu

/tmp/ipykernel_13/3462763141.py:52: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["user_created"] = pd.to_datetime(
/tmp/ipykernel_13/3462763141.py:110: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[DATE_TWEET] = pd.to_datetime(df[DATE_TWEET], errors="coerce").dt.tz_localize(None)
/tmp/ipykernel_13/3462763141.py:111: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[DATE_USER]  = pd.to_datetime(df[DATE_USER], errors="coerce").dt.tz_localize(None)
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `spar

X_user_test shape: (29736, 158)


/tmp/ipykernel_13/3462763141.py:52: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["user_created"] = pd.to_datetime(



🎉 Submission saved to: /kaggle/working/submission_user_level_xgb.csv
  ID  label
0  0      0
1  2      1
2  4      1
3  8      1
4  9      0


In [2]:
import pandas as pd
import numpy as np
import json
import hashlib
from tqdm import tqdm

import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.pipeline import Pipeline

tqdm.pandas()

# ==========================================================
# 1) JSONL LOADER
# ==========================================================
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


# ==========================================================
# 2) DESCRIPTION CLEANING + HASH USER ID
# ==========================================================
def extract_description(user_field):
    if isinstance(user_field, dict):
        d = user_field.get("description", "")
        return d if isinstance(d, str) else ""
    return ""

def clean_desc(s: str):
    s = s.strip().lower()
    return " ".join(s.split())

def to_day_index(dt):
    if pd.isna(dt):
        return -1
    ref = pd.Timestamp("2006-03-01")
    return (dt - ref).days

def build_pseudo_user_id(df):
    df = df.copy()

    # Description clean
    df["description_clean"] = df["user"].apply(extract_description).apply(clean_desc)

    # Parse account creation date
    df["user_created"] = pd.to_datetime(
        df["user"].apply(lambda u: u.get("created_at") if isinstance(u, dict) else None),
        errors="coerce"
    ).dt.tz_localize(None)

    df["creation_day"] = df["user_created"].apply(to_day_index)

    # Hash desc + day
    def hash_user(desc, day):
        return hashlib.md5(f"{desc}__{day}".encode()).hexdigest()

    df["pseudo_user_id"] = df.apply(
        lambda x: hash_user(x["description_clean"], x["creation_day"]),
        axis=1
    )

    return df


# ==========================================================
# 3) TWEET-LEVEL FEATURE EXTRACTION
# ==========================================================
def extract_all_features_tweet_level(df):
    """
    Tweet-level :
    - user.* explosé
    - dates -> month_index + user_month_index + tweet_age_months
    - couleurs -> booléens (is_default)
    - catégorielles -> one-hot (lang + translator_type)
    - numériques / bool nettoyés
    - nouvelles features : longueur du tweet, engagement, ratio fav/RT
    """

    df = df.copy()

    # Explode user.*
    user_df = pd.json_normalize(df["user"])
    user_df.columns = [f"user.{c}" for c in user_df.columns]
    df = pd.concat([df, user_df], axis=1)

    # Longueur du tweet + engagement
    df["text"] = df["text"].astype(str)
    df["tweet_length"] = df["text"].str.len()
    df["engagement"] = df["favorite_count"].fillna(0) + df["retweet_count"].fillna(0)
    df["fav_to_rt_ratio"] = df["favorite_count"] / (df["retweet_count"] + 1)

    NUMERIC_COLS = [
        "retweet_count", "favorite_count", "quote_count", "reply_count",
        "possibly_sensitive",
        "user.listed_count", "user.favourites_count", "user.statuses_count",
        "tweet_length", "engagement", "fav_to_rt_ratio"
    ]

    BOOL_COLS = [
        "retweeted", "is_quote_status", "favorited", "truncated",
        "user.default_profile_image", "user.protected", "user.geo_enabled",
        "user.is_translator", "user.contributors_enabled",
        "user.profile_use_background_image", "user.profile_background_tile",
    ]

    CAT_COLS = ["user.translator_type", "user.lang"]

    # Dates
    DATE_TWEET = "created_at"
    DATE_USER  = "user.created_at"

    df[DATE_TWEET] = pd.to_datetime(df[DATE_TWEET], errors="coerce").dt.tz_localize(None)
    df[DATE_USER]  = pd.to_datetime(df[DATE_USER], errors="coerce").dt.tz_localize(None)

    REF_YEAR = 2006
    REF_MONTH = 3

    def to_month_index(dt):
        if pd.isna(dt): 
            return -1
        return (dt.year - REF_YEAR) * 12 + (dt.month - REF_MONTH)

    df["tweet_month_index"] = df[DATE_TWEET].apply(to_month_index)
    df["user_month_index"]  = df[DATE_USER].apply(to_month_index)
    df["tweet_age_months"]  = df["tweet_month_index"] - df["user_month_index"]

    DATE_FEATURES = ["tweet_month_index", "user_month_index", "tweet_age_months"]

    # Default colors -> bool
    default_colors = {
        "user.profile_link_color": "1DA1F2",
        "user.profile_background_color": "F5F8FA",
        "user.profile_sidebar_border_color": "C0DEED",
        "user.profile_text_color": "333333",
        "user.profile_sidebar_fill_color": "DDEEF6",
    }

    for col, default_val in default_colors.items():
        df[col + "_is_default"] = (df[col] == default_val).astype(float)
        BOOL_COLS.append(col + "_is_default")

    df = df.drop(columns=list(default_colors.keys()), errors="ignore")

    # Clean numeric / bool
    for col in NUMERIC_COLS + DATE_FEATURES:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    for col in BOOL_COLS:
        df[col] = df[col].astype(float).fillna(0)

    # Categorical
    for col in CAT_COLS:
        df[col] = df[col].astype(str).fillna("missing")

    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    X_cat_arr = encoder.fit_transform(df[CAT_COLS])
    X_cat = pd.DataFrame(
        X_cat_arr,
        columns=encoder.get_feature_names_out(CAT_COLS),
        index=df.index
    )

    X = pd.concat([
        df[NUMERIC_COLS + DATE_FEATURES + BOOL_COLS],
        X_cat
    ], axis=1)

    return X


# ==========================================================
# 4) USER-LEVEL AGGREGATION (ENRICHIE)
# ==========================================================
def aggregate_user_level(df, X_meta):
    """
    df      = df_train ou df_test (incluant pseudo_user_id + BERT scores)
    X_meta  = tweet-level features

    Ajoute :
    - stats globales (mean/std/min/max/median)
    - temps moyen entre tweets, min/max, std
    - entropies temporelles (heure, jour)
    - corrélation entre bert_score et bert_descr_score
    - taux de désaccord entre BERT texte / descr
    - n_tweets, log_n_tweets
    """

    df = df.copy()
    X = X_meta.copy()

    # pseudo_user_id
    X["pseudo_user_id"] = df["pseudo_user_id"].values

    # Dates de tweet
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce").dt.tz_localize(None)

    # Ajout BERT dans X pour agrégation
    for col in ["bert_score", "bert_conf", "bert_descr_score", "bert_descr_conf"]:
        if col in df.columns:
            X[col] = df[col].values

    if "pred" in df.columns:
        X["bert_pred_text"] = df["pred"].values
    if "label_pred" in df.columns:
        X["bert_pred_desc"] = df["label_pred"].values

    # 1) Base numeric aggregation
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    num_cols = [c for c in num_cols if c not in ["pseudo_user_id"]]

    agg_funcs = ["mean", "std", "min", "max", "median"]
    grouped = X.groupby("pseudo_user_id")[num_cols].agg(agg_funcs)
    grouped.columns = [f"{col}_{stat}" for (col, stat) in grouped.columns]

    users = grouped.copy()

    # 2) TEMPORAL FEATURES (time between tweets + entropies)
    temporal = {}
    for uid, frame in df.groupby("pseudo_user_id"):
        timestamps = frame["created_at"].dropna().sort_values()

        if len(timestamps) > 1:
            diffs = timestamps.diff().dt.total_seconds().dropna() / 3600.0  # en heures
            mean_diff = diffs.mean()
            std_diff  = diffs.std()
            min_diff  = diffs.min()
            max_diff  = diffs.max()
        else:
            mean_diff = std_diff = min_diff = max_diff = 0.0

        if timestamps.notna().any():
            hour_counts = timestamps.dt.hour.value_counts(normalize=True)
            weekday_counts = timestamps.dt.weekday.value_counts(normalize=True)

            hour_entropy = -(hour_counts * np.log(hour_counts + 1e-12)).sum()
            weekday_entropy = -(weekday_counts * np.log(weekday_counts + 1e-12)).sum()
        else:
            hour_entropy = 0.0
            weekday_entropy = 0.0

        temporal[uid] = {
            "mean_time_between_tweets": mean_diff,
            "std_time_between_tweets": std_diff,
            "min_time_between_tweets": min_diff,
            "max_time_between_tweets": max_diff,
            "tweet_entropy_hour": hour_entropy,
            "tweet_entropy_weekday": weekday_entropy
        }

    users = users.join(pd.DataFrame.from_dict(temporal, orient="index"))

    # 3) Corrélation score BERT texte vs descr
    if "bert_score" in X.columns and "bert_descr_score" in X.columns:
        corr_dict = {}
        for uid, frame in X.groupby("pseudo_user_id"):
            if len(frame) > 1:
                corr = np.corrcoef(frame["bert_score"], frame["bert_descr_score"])[0, 1]
                if np.isnan(corr):
                    corr = 0.0
                corr_dict[uid] = {"bert_score_correlation": corr}
            else:
                corr_dict[uid] = {"bert_score_correlation": 0.0}
        users = users.join(pd.DataFrame.from_dict(corr_dict, orient="index"))

    # 4) Taux de désaccord BERT texte vs descr
    if "bert_pred_text" in X.columns and "bert_pred_desc" in X.columns:
        switch_dict = {}
        for uid, frame in X.groupby("pseudo_user_id"):
            if len(frame) > 0:
                switches = (frame["bert_pred_text"] != frame["bert_pred_desc"]).mean()
            else:
                switches = 0.0
            switch_dict[uid] = {"bert_switch_rate": switches}
        users = users.join(pd.DataFrame.from_dict(switch_dict, orient="index"))

    # 5) n_tweets
    tweet_counts = X.groupby("pseudo_user_id").size()
    users["n_tweets"] = tweet_counts
    users["log_n_tweets"] = np.log1p(tweet_counts)

    users = users.fillna(0)

    # 6) Labels (si train)
    if "label" in df.columns:
        labels = df.groupby("pseudo_user_id")["label"].first()
        return users, labels
    else:
        return users


def build_user_level_dataset(df, mode="train"):
    """
    df : raw df with tweets (+ BERT merged)
    mode: "train" or "test"
    """
    df = build_pseudo_user_id(df)
    X_meta = extract_all_features_tweet_level(df)

    if mode == "train":
        X_user, y_user = aggregate_user_level(df, X_meta)
        return X_user, y_user
    else:
        X_user = aggregate_user_level(df, X_meta)
        return X_user


# ==========================================================
# 5) MAIN PIPELINE: MERGE BERT + BUILD USER DATASETS
# ==========================================================

# -------------------- LOAD TRAIN --------------------
print("Loading TRAIN jsonl...")
df_train = load_jsonl("/kaggle/input/influencers-or-observers-predicting-social-roles/Kaggle2025/train.jsonl")
df_train["challenge_id"] = df_train["challenge_id"].astype(str)

# -------------------- MERGE BERT OOF (TEXT + DESC) --------------------
print("Merging BERT OOF (TEXT + DESCRIPTION) on TRAIN...")

# Charger OOF
bert_oof_text = pd.read_csv("/kaggle/input/bert-with-proper-oof/bert_train_oof.csv")
bert_oof_desc = pd.read_csv("/kaggle/input/bert-with-proper-oof/bert_userdesc_train_oof.csv")

bert_oof_text["challenge_id"] = bert_oof_text["challenge_id"].astype(str)
bert_oof_desc["challenge_id"] = bert_oof_desc["challenge_id"].astype(str)

# ---------------------------
# TEXT BERT  (colonnes connues)
# challenge_id, bert_pred, prob_0, prob_1, bert_conf, label
# ---------------------------
bert_oof_text = bert_oof_text.rename(columns={
    "bert_pred": "bert_pred_text",
    "prob_1": "bert_score",
    "bert_conf": "bert_conf"
})[["challenge_id", "bert_score", "bert_conf", "bert_pred_text"]]

# ---------------------------
# DESCRIPTION BERT (colonnes connues)
# challenge_id, pred, prob_0, prob_1, confidence, label
# ---------------------------
bert_oof_desc = bert_oof_desc.rename(columns={
    "pred": "bert_pred_desc",
    "prob_1": "bert_descr_score",
    "confidence": "bert_descr_conf"
})[["challenge_id", "bert_descr_score", "bert_descr_conf", "bert_pred_desc"]]

# Merge sur TRAIN
df_train = df_train.merge(bert_oof_text, on="challenge_id", how="left")
df_train = df_train.merge(bert_oof_desc, on="challenge_id", how="left")

print("Missing TEXT OOF:", df_train["bert_score"].isna().sum())
print("Missing DESC OOF:", df_train["bert_descr_score"].isna().sum())

# Garder uniquement les lignes où on possède bien OOF
df_train_oof = df_train.dropna(subset=["bert_score", "bert_descr_score"]).reset_index(drop=True)
print("Train OOF subset shape:", df_train_oof.shape)


# -------------------- BUILD USER-LEVEL TRAIN DATASET --------------------
print("\nBuilding USER-LEVEL TRAIN dataset...")
X_user_train, y_user_train = build_user_level_dataset(df_train_oof, mode="train")

print("X_user_train shape:", X_user_train.shape)
print("y_user_train shape:", y_user_train.shape)


# ==========================================================
# 6) 5-FOLD CV : XGBOOST, RANDOM FOREST, RIDGE
# ==========================================================
print("\n===== 5-FOLD CV (USER-LEVEL) =====")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X_np = X_user_train.values
y_np = y_user_train.values

# --- XGBoost params ---
xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "eta": 0.03,
    "max_depth": 6,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
}

xgb_accuracies = []
xgb_best_iters = []

rf_accuracies = []
ridge_accuracies = []

fold_id = 1
for train_idx, val_idx in skf.split(X_np, y_np):
    print(f"\n--- Fold {fold_id} ---")

    X_tr, X_val = X_np[train_idx], X_np[val_idx]
    y_tr, y_val = y_np[train_idx], y_np[val_idx]

    # ---------- XGBOOST ----------
    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dval   = xgb.DMatrix(X_val, label=y_val)

    xgb_model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=1000,
        evals=[(dval, "val")],
        early_stopping_rounds=50,
        verbose_eval=False
    )

    xgb_best_iters.append(xgb_model.best_iteration)
    y_val_pred = (xgb_model.predict(dval) > 0.5).astype(int)
    acc_xgb = accuracy_score(y_val, y_val_pred)
    xgb_accuracies.append(acc_xgb)
    print(f"XGBoost ACC fold {fold_id}: {acc_xgb:.4f}")

    # ---------- RANDOM FOREST ----------
    rf = RandomForestClassifier(
        n_estimators=400,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=42
    )
    rf.fit(X_tr, y_tr)
    y_val_pred_rf = rf.predict(X_val)
    acc_rf = accuracy_score(y_val, y_val_pred_rf)
    rf_accuracies.append(acc_rf)
    print(f"RandomForest ACC fold {fold_id}: {acc_rf:.4f}")

    # ---------- RIDGE ----------
    ridge = Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", RidgeClassifier(alpha=1.0))
    ])
    ridge.fit(X_tr, y_tr)
    y_val_pred_ridge = ridge.predict(X_val)
    acc_ridge = accuracy_score(y_val, y_val_pred_ridge)
    ridge_accuracies.append(acc_ridge)
    print(f"RidgeClassifier ACC fold {fold_id}: {acc_ridge:.4f}")

    fold_id += 1

print("\n===== CV SUMMARY =====")
print("XGBoost ACC per fold:", xgb_accuracies)
print("XGBoost mean ACC:", np.mean(xgb_accuracies))
print("RandomForest ACC per fold:", rf_accuracies)
print("RandomForest mean ACC:", np.mean(rf_accuracies))
print("Ridge ACC per fold:", ridge_accuracies)
print("Ridge mean ACC:", np.mean(ridge_accuracies))

mean_best_iter = int(np.mean(xgb_best_iters))
print("\nMean best iteration for XGBoost:", mean_best_iter)


# ==========================================================
# 7) TRAIN FINAL XGBOOST ON ALL USER-LEVEL TRAIN
# ==========================================================
print("\nTraining FINAL XGBoost on all user-level train...")

dtrain_full = xgb.DMatrix(X_user_train.values, label=y_user_train.values)
final_model = xgb.train(
    xgb_params,
    dtrain_full,
    num_boost_round=mean_best_iter
)

print("Final XGBoost trained with", mean_best_iter, "trees.")


# ==========================================================
# 8) TEST PIPELINE: MERGE BERT + USER-LEVEL + PREDICTIONS
# ==========================================================
print("\nLoading TEST jsonl...")
df_test = load_jsonl("/kaggle/input/influencers-or-observers-predicting-social-roles/Kaggle2025/kaggle_test.jsonl")
df_test["challenge_id"] = df_test["challenge_id"].astype(str)

print("Merging BERT TEST predictions...")

bert_test_text = pd.read_csv("/kaggle/input/bert-with-proper-oof/bert_test_predictions.csv")
bert_test_desc = pd.read_csv("/kaggle/input/bert-with-proper-oof/bert_userdesc_test_predictions.csv")

bert_test_text["challenge_id"] = bert_test_text["challenge_id"].astype(str)
bert_test_desc["challenge_id"] = bert_test_desc["challenge_id"].astype(str)

bert_test_text = bert_test_text.rename(columns={
    "bert_pred": "bert_pred_text",
    "prob_1": "bert_score",
    "bert_conf": "bert_conf"
})[["challenge_id", "bert_score", "bert_conf", "bert_pred_text"]]

bert_test_desc = bert_test_desc.rename(columns={
    "pred": "bert_pred_desc",
    "prob_1": "bert_descr_score",
    "confidence": "bert_descr_conf"
})[["challenge_id", "bert_descr_score", "bert_descr_conf", "bert_pred_desc"]]

df_test = df_test.merge(bert_test_text, on="challenge_id", how="left")
df_test = df_test.merge(bert_test_desc, on="challenge_id", how="left")


print("Test shape after BERT merge:", df_test.shape)

print("\nBuilding USER-LEVEL TEST dataset...")
X_user_test = build_user_level_dataset(df_test, mode="test")
print("X_user_test shape:", X_user_test.shape)

dtest_users = xgb.DMatrix(X_user_test.values)
user_pred_prob = final_model.predict(dtest_users)
user_pred = (user_pred_prob > 0.5).astype(int)

df_user_pred = pd.DataFrame({
    "pseudo_user_id": X_user_test.index,
    "user_label": user_pred
})

# Recalcul des pseudo_user_id sur df_test pour mapping
df_test_with_uid = build_pseudo_user_id(df_test)

df_test_with_uid = df_test_with_uid.merge(df_user_pred, on="pseudo_user_id", how="left")

submission = pd.DataFrame({
    "ID": df_test_with_uid["challenge_id"],
    "label": df_test_with_uid["user_label"].astype(int)
})

save_path = "/kaggle/working/submission_user_level_xgb.csv"
submission.to_csv(save_path, index=False)

print("\n🎉 Submission saved to:", save_path)
print(submission.head())


Loading TRAIN jsonl...
Merging BERT OOF (TEXT + DESCRIPTION) on TRAIN...
Missing TEXT OOF: 116120
Missing DESC OOF: 116120
Train OOF subset shape: (38794, 43)

Building USER-LEVEL TRAIN dataset...


/tmp/ipykernel_13/559424522.py:55: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["user_created"] = pd.to_datetime(
/tmp/ipykernel_13/559424522.py:121: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[DATE_TWEET] = pd.to_datetime(df[DATE_TWEET], errors="coerce").dt.tz_localize(None)
/tmp/ipykernel_13/559424522.py:122: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[DATE_USER]  = pd.to_datetime(df[DATE_USER], errors="coerce").dt.tz_localize(None)
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_

X_user_train shape: (12722, 199)
y_user_train shape: (12722,)

===== 5-FOLD CV (USER-LEVEL) =====

--- Fold 1 ---
XGBoost ACC fold 1: 0.8267
RandomForest ACC fold 1: 0.8275
RidgeClassifier ACC fold 1: 0.7745

--- Fold 2 ---
XGBoost ACC fold 2: 0.8550
RandomForest ACC fold 2: 0.8519
RidgeClassifier ACC fold 2: 0.8094

--- Fold 3 ---
XGBoost ACC fold 3: 0.8373
RandomForest ACC fold 3: 0.8471
RidgeClassifier ACC fold 3: 0.7936

--- Fold 4 ---
XGBoost ACC fold 4: 0.8506
RandomForest ACC fold 4: 0.8396
RidgeClassifier ACC fold 4: 0.7889

--- Fold 5 ---
XGBoost ACC fold 5: 0.8510
RandomForest ACC fold 5: 0.8534
RidgeClassifier ACC fold 5: 0.7995

===== CV SUMMARY =====
XGBoost ACC per fold: [0.8267190569744597, 0.8550098231827112, 0.8372641509433962, 0.85062893081761, 0.8510220125786163]
XGBoost mean ACC: 0.8441287948993587
RandomForest ACC per fold: [0.8275049115913556, 0.8518664047151278, 0.8470911949685535, 0.839622641509434, 0.8533805031446541]
RandomForest mean ACC: 0.843893131185825
Ri

/tmp/ipykernel_13/559424522.py:55: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["user_created"] = pd.to_datetime(
/tmp/ipykernel_13/559424522.py:121: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[DATE_TWEET] = pd.to_datetime(df[DATE_TWEET], errors="coerce").dt.tz_localize(None)
/tmp/ipykernel_13/559424522.py:122: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[DATE_USER]  = pd.to_datetime(df[DATE_USER], errors="coerce").dt.tz_localize(None)
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_

X_user_test shape: (29736, 199)


/tmp/ipykernel_13/559424522.py:55: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["user_created"] = pd.to_datetime(



🎉 Submission saved to: /kaggle/working/submission_user_level_xgb.csv
  ID  label
0  0      0
1  2      1
2  4      1
3  8      1
4  9      0


In [3]:
import itertools
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import numpy as np
import time
print("X_user_train shape:", X_user_train.shape)
print("y_user_train distribution:\n", y_user_train.value_counts(normalize=True))

# ============================================================
# GRID SEARCH XGBOOST — USER LEVEL
# ============================================================

# Un grid raisonnable mais puissant
param_grid = {
    "n_estimators": [600, 1000, 2000],
    "max_depth": [6, 8, 10],
    "learning_rate": [0.01, 0.03, 0.05],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.7, 0.9],
    "gamma": [0, 1]
}

param_list = list(itertools.product(
    param_grid["n_estimators"],
    param_grid["max_depth"],
    param_grid["learning_rate"],
    param_grid["subsample"],
    param_grid["colsample_bytree"],
    param_grid["gamma"]
))

total_configs = len(param_list)
print(f"\n🔍 Total configurations to test: {total_configs}")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_score = 0
best_params = None
results = []

# ============================================================
# GRID SEARCH LOOP (MANUAL FOR VERBOSE FEEDBACK)
# ============================================================
for idx, (n_est, depth, lr, subs, col, gamma) in enumerate(param_list, start=1):

    params = {
        "n_estimators": n_est,
        "max_depth": depth,
        "learning_rate": lr,
        "subsample": subs,
        "colsample_bytree": col,
        "gamma": gamma,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1
    }

    print(f"\n====================================")
    print(f"🔧 Config {idx}/{total_configs} : {params}")
    print("====================================")

    fold_scores = []
    fold_id = 1
    start_time = time.time()

    for train_idx, val_idx in skf.split(X_user_train, y_user_train):

        X_tr, X_val = X_user_train.iloc[train_idx], X_user_train.iloc[val_idx]
        y_tr, y_val = y_user_train.iloc[train_idx], y_user_train.iloc[val_idx]

        model = XGBClassifier(**params)

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        preds = model.predict(X_val)
        acc = accuracy_score(y_val, preds)
        fold_scores.append(acc)

        print(f"  Fold {fold_id} ACC = {acc:.4f}")
        fold_id += 1

    mean_acc = np.mean(fold_scores)
    elapsed = time.time() - start_time

    print(f"➡️  Mean ACC for this config: {mean_acc:.4f} (time {elapsed:.1f}s)")

    results.append((params, mean_acc))

    if mean_acc > best_score:
        best_score = mean_acc
        best_params = params
        print(f"🔥 NEW BEST SCORE: {best_score:.4f}")
        print(f"🔥 NEW BEST PARAMS: {best_params}")

print("\n====================================")
print("🏁 GRID SEARCH COMPLETE")
print("====================================")
print(f"🥇 BEST CV SCORE: {best_score:.4f}")
print("🥇 BEST PARAMS:", best_params)

X_user_train shape: (12722, 199)
y_user_train distribution:
 label
0    0.537966
1    0.462034
Name: proportion, dtype: float64

🔍 Total configurations to test: 216

🔧 Config 1/216 : {'n_estimators': 600, 'max_depth': 6, 'learning_rate': 0.01, 'subsample': 0.8, 'colsample_bytree': 0.7, 'gamma': 0, 'objective': 'binary:logistic', 'eval_metric': 'logloss', 'tree_method': 'hist', 'random_state': 42, 'n_jobs': -1}
  Fold 1 ACC = 0.8279
  Fold 2 ACC = 0.8554
  Fold 3 ACC = 0.8432
  Fold 4 ACC = 0.8522
  Fold 5 ACC = 0.8526
➡️  Mean ACC for this config: 0.8463 (time 49.1s)
🔥 NEW BEST SCORE: 0.8463
🔥 NEW BEST PARAMS: {'n_estimators': 600, 'max_depth': 6, 'learning_rate': 0.01, 'subsample': 0.8, 'colsample_bytree': 0.7, 'gamma': 0, 'objective': 'binary:logistic', 'eval_metric': 'logloss', 'tree_method': 'hist', 'random_state': 42, 'n_jobs': -1}

🔧 Config 2/216 : {'n_estimators': 600, 'max_depth': 6, 'learning_rate': 0.01, 'subsample': 0.8, 'colsample_bytree': 0.7, 'gamma': 1, 'objective': 'bin